# 08 — SHAP Explainability

Explain which drivers push customers toward churn. Prefer **TabNet**-related explanations via model importances + Kernel/Permutation SHAP on a sample; if too heavy, fall back to **TreeExplainer** on the Random Forest with an explicit note.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42


In [2]:
import json
import warnings
warnings.filterwarnings("ignore")
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import shap

X_train = pd.read_csv(DATA_PROCESSED / "X_train.csv")
X_test = pd.read_csv(DATA_PROCESSED / "X_test.csv")
y_test = pd.read_csv(DATA_PROCESSED / "y_test.csv").squeeze()
ids_test = pd.read_csv(DATA_PROCESSED / "id_test.csv").squeeze()
winner = json.loads((DATA_PROCESSED / "nia_feature_selection_winner.json").read_text(encoding="utf-8"))
win_feats = winner["winning_features"]

rf = joblib.load(MODELS_DIR / "rf_baseline.joblib")
print("Loaded RF baseline; winning features:", len(win_feats))

Loaded RF baseline; winning features: 17


## Strategy

TabNet SHAP (KernelExplainer) is expensive. We:

1. Show TabNet's built-in global feature importance (from notebook 07 artefact).
2. Run **TreeExplainer** on Random Forest for rich global/local SHAP plots on the same winning feature set (fast, exact for trees).
3. Translate plots into business language for retention.

In [3]:
imp = pd.read_csv(DATA_PROCESSED / "07_tabnet_feature_importance.csv")
fig, ax = plt.subplots(figsize=(8, max(3, len(imp) * 0.25)))
ax.barh(imp["feature"], imp["importance"], color="#2a9d8f")
ax.invert_yaxis()
ax.set_title("Global importance — TabNet (winning subset)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "08_tabnet_global_importance.png", dpi=150)
plt.show()
imp.head(10)

,feature,importance
0,MaritalStatus_Single,0.169171
1,Tenure,0.164117
2,CityTier,0.093919
3,SatisfactionScore,0.088097
4,OrderAmountHikeFromlastYear,0.067087
5,PreferredLoginDevice_Computer,0.061134
6,Complain,0.055697
7,PreferedOrderCat_Laptop & Accessory,0.054181
8,NumberOfAddress,0.051583
9,PreferedOrderCat_Mobile Phone,0.036043


In [4]:
# SHAP on RF using winning features (aligned with NIA subset)
Xtr = X_train[win_feats]
Xte = X_test[win_feats]

# Refit a small RF on winning features for TreeExplainer alignment
from sklearn.ensemble import RandomForestClassifier
rf_w = RandomForestClassifier(
    n_estimators=100, max_depth=8, min_samples_leaf=5,
    class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
)
y_train = pd.read_csv(DATA_PROCESSED / "y_train.csv").squeeze()
rf_w.fit(Xtr, y_train)
joblib.dump(rf_w, MODELS_DIR / "rf_winning_subset.joblib")

explainer = shap.TreeExplainer(rf_w)
# Sample for summary plot speed
sample_idx = np.random.default_rng(RANDOM_SEED).choice(len(Xte), size=min(400, len(Xte)), replace=False)
X_sample = Xte.iloc[sample_idx]
shap_values = explainer.shap_values(X_sample)

# Binary classifier: shap_values may be list [class0, class1]
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values
    # If 3D (n, features, classes)
    if getattr(sv, "ndim", 2) == 3:
        sv = sv[:, :, 1]

plt.figure()
shap.summary_plot(sv, X_sample, show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_shap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure()
shap.summary_plot(sv, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_shap_summary_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [5]:
# Dependence plot for top feature
mean_abs = np.abs(sv).mean(axis=0)
top_i = int(np.argmax(mean_abs))
top_feat = win_feats[top_i]
print("Top SHAP feature:", top_feat)

plt.figure()
shap.dependence_plot(top_feat, sv, X_sample, show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_shap_dependence_top.png", dpi=150, bbox_inches="tight")
plt.show()

Top SHAP feature: Tenure


In [6]:
# Local explanations: one churned, one retained
proba = rf_w.predict_proba(Xte)[:, 1]
churned_i = int(np.argmax(proba))
retained_i = int(np.argmin(proba))

for label, idx in [("churned", churned_i), ("retained", retained_i)]:
    row = Xte.iloc[[idx]]
    print(f"\n=== Local example: {label} ===")
    print("CustomerID:", ids_test.iloc[idx], "P(churn)=", proba[idx], "Actual=", y_test.iloc[idx])
    sv_row = explainer.shap_values(row)
    if isinstance(sv_row, list):
        sv1 = sv_row[1][0]
    else:
        sv1 = sv_row[0]
        if getattr(sv1, "ndim", 1) > 1:
            sv1 = sv1[:, 1] if sv1.shape[-1] == 2 else sv1
    # Waterfall via matplotlib bar of contributions
    order = np.argsort(np.abs(sv1))[::-1][:12]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh([win_feats[j] for j in order][::-1], sv1[order][::-1], color="#e76f51")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"Local SHAP — {label} customer {ids_test.iloc[idx]}")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"08_shap_local_{label}.png", dpi=150)
    plt.show()


=== Local example: churned ===
CustomerID: 54618 P(churn)= 0.9788384259303338 Actual= 1

=== Local example: retained ===
CustomerID: 52586 P(churn)= 0.020229168051375136 Actual= 0


## Business translation

| Signal pattern | Business meaning | Retention action |
|----------------|------------------|------------------|
| High positive SHAP on Complain / UnhappyComplain | Service failure recently experienced | Priority support callback; resolve ticket before promo |
| Low Tenure pushing churn | New customers still fragile | Onboarding journey, first-order success guarantee |
| Low Cashback / engagement | Weak loyalty loop | Targeted cashback or category-relevant coupon |
| High DaySinceLastOrder | Dormancy risk | Win-back campaign within recency window |
| Device / payment / city effects | Channel friction | Fix UX for high-risk channels; do not over-discount demographics |

**Note:** Global TabNet importances and RF SHAP should be read together. Tree SHAP is exact for the RF surrogate; TabNet importance reflects the attentive neural predictor used for final scores.

**Next:** Notebook `09` — business insights synthesis + README polish.